In [16]:
import pandas as pd

from eavs.config import RAW_DATA_DIR, INTERIM_DATA_DIR

In [17]:
eavs_2022: pd.DataFrame = pd.read_excel(
    RAW_DATA_DIR / "2022" / "1.1" / "2022_EAVS_for_Public_Release_V1.1.xlsx", engine="calamine", dtype_backend="pyarrow"
)
eavs_2022.head()

,FIPSCode,Jurisdiction_Name,State_Full,State_Abbr,A1a,A1b,A1c,A1Comments,A2a,A2b,...,F9d_4,F9d_5,F5_F9Comments,F10a,F10b,F10c,F10d,F10e,F10Comments,F11
0,100100000,AUTAUGA COUNTY,ALABAMA,AL,43488,39027,4461,<NA>,Does not apply,Does not apply,...,Valid skip,Valid skip,<NA>,Precinct or polling location,Precinct or polling location,Central location,Does not apply,Central location,<NA>,<NA>
1,100300000,BALDWIN COUNTY,ALABAMA,AL,189028,164133,24895,<NA>,Does not apply,Does not apply,...,Valid skip,Valid skip,<NA>,Precinct or polling location,Precinct or polling location,Central location,Does not apply,Central location,<NA>,<NA>
2,100500000,BARBOUR COUNTY,ALABAMA,AL,17270,15203,2067,<NA>,Does not apply,Does not apply,...,Valid skip,Valid skip,<NA>,Precinct or polling location,Precinct or polling location,Central location,Does not apply,Central location,<NA>,<NA>
3,100700000,BIBB COUNTY,ALABAMA,AL,14741,13804,937,<NA>,Does not apply,Does not apply,...,Valid skip,Valid skip,<NA>,Precinct or polling location,Precinct or polling location,Central location,Does not apply,Central location,<NA>,<NA>
4,100900000,BLOUNT COUNTY,ALABAMA,AL,41794,39011,2783,<NA>,Does not apply,Does not apply,...,Valid skip,Valid skip,<NA>,Precinct or polling location,Precinct or polling location,Central location,Does not apply,Central location,<NA>,<NA>


In [18]:
# A3a is total voter registrations, and A3e is total rejected registrations.
columns = ["State_Full", "A3a", "A3e"]
eavs_2022 = eavs_2022.filter(items=columns)

# Remove all counties where there is no data for either of the columns
for col in columns:
    eavs_2022 = eavs_2022[eavs_2022[col] != "Data not available"]
    eavs_2022 = eavs_2022[eavs_2022[col] != "Does not apply"]

# Convert total voter registrations and total rejected registrations to numbers instead of strings.
eavs_2022["A3a"] = pd.to_numeric(eavs_2022["A3a"], dtype_backend="pyarrow")
eavs_2022["A3e"] = pd.to_numeric(eavs_2022["A3e"], dtype_backend="pyarrow")

# In order to get state data instead of county data, aggregate all remaining counties belonging
# to a state and summing their total registration and rejected registration values.
eavs_2022 = eavs_2022.groupby("State_Full").agg({"A3a": "sum", "A3e": "sum"})

# Rejected % is Total Rejected Registrations / Total Voter Registrations
eavs_2022["Reject_%"] = eavs_2022["A3e"] / eavs_2022["A3a"] * 100

# Rename and filter columns to make similar to Looker Studio table
eavs_2022 = eavs_2022.rename(columns={"A3a": "Reject_Count"})
eavs_2022 = eavs_2022.filter(items=["Reject_Count", "Reject_%"])

eavs_2022.to_csv(INTERIM_DATA_DIR / "eavs_2022.csv")